In [ ]:
import ee

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='my-ee-project-505909')

In [ ]:
AOI = ee.Geometry.Rectangle([103.80, 1.20, 103.90, 1.28])  # near Singapore
START_DATE, END_DATE = "2026-09-01", "2026-10-01"

GUARD_RADIUS = 5        # pixels: hole in the donut (ship zone, ignored)
OUTER_RADIUS = 20       # pixels: outer edge of the donut (background)
Z_THRESHOLD = 5         # how many std-devs above background = "surprising"
DB_FLOOR = -18          # absolute brightness floor in dB
MIN_PIXELS = 2          # smallest cluster accepted as a ship
MAX_PIXELS = 150        # largest cluster accepted as a ship
COAST_BUFFER_M = 500    # keep this far from the shoreline
FIXED_OBJ_DB = -15      # median brightness above this = fixed structure
N_SCENES_FOR_MEDIAN = 6


In [ ]:
# --- AOI and image ---
import geemap
from google.colab import output
output.enable_custom_widget_manager()
Map = geemap.Map()
Map.centerObject(AOI, 11);
Map.add_basemap("SATELLITE")
Map



Map(center=[1.2400002705199196, 103.84999999999863], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
)


In [ ]:
first = ee.Image(s1.first())
img = first.select("VH").clip(AOI)  # values in dB

print("Scenes found:", s1.size().getInfo())
print("Acquisition time (UTC):",
      ee.Date(first.get("system:time_start")).format("YYYY-MM-dd HH:mm").getInfo())



Scenes found: 5
Acquisition time (UTC): 2026-09-03 11:24


In [ ]:
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(90)
img = img.updateMask(water.focalMin(radius=COAST_BUFFER_M, units="meters"))


In [ ]:
# ---------------------------------------------------------------
# CFAR: donut kernel, ONE-PASS background stats, threshold
# ---------------------------------------------------------------
import math
size = 2 * OUTER_RADIUS + 1
weights = [
    [1 if GUARD_RADIUS < math.hypot(x, y) <= OUTER_RADIUS else 0
     for x in range(-OUTER_RADIUS, OUTER_RADIUS + 1)]
    for y in range(-OUTER_RADIUS, OUTER_RADIUS + 1)
]
kernel = ee.Kernel.fixed(width=size, height=size, weights=weights)

stats = img.reduceNeighborhood(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    kernel=kernel,
)
bg_mean = stats.select("VH_mean")
bg_std = stats.select("VH_stdDev")

z = img.subtract(bg_mean).divide(bg_std)
raw_ships = z.gt(Z_THRESHOLD).And(img.gt(DB_FLOOR)).selfMask()



In [ ]:
# ---------------------------------------------------------------
# Filter 1: cluster size
# ---------------------------------------------------------------
cluster_size = raw_ships.connectedPixelCount(maxSize=200, eightConnected=True)
ships = raw_ships.updateMask(
    cluster_size.gte(MIN_PIXELS).And(cluster_size.lte(MAX_PIXELS))
)


In [ ]:
# ---------------------------------------------------------------
# Filter 2: fixed structures (median of only a few scenes)
# ---------------------------------------------------------------
median_vh = s1.select("VH").limit(N_SCENES_FOR_MEDIAN).median().clip(AOI)
fixed_objects = median_vh.gt(FIXED_OBJ_DB).focalMax(radius=30, units="meters")
ships = ships.updateMask(fixed_objects.Not())

In [ ]:
# ---------------------------------------------------------------
# Vectorize (tileScale splits the work into smaller chunks)
# ---------------------------------------------------------------
detections = ships.rename("ship").reduceToVectors(
    geometry=AOI,
    scale=10,
    geometryType="centroid",
    maxPixels=int(1e9),
    reducer=ee.Reducer.countEvery(),
    tileScale=4,
)


In [ ]:
features = detections.getInfo()["features"]
coords = [f["geometry"]["coordinates"] for f in features]  # [lon, lat]
print("Detections:", len(coords))
print("First 5 (lon, lat):", coords[:5])

Detections: 27
First 5 (lon, lat): [[103.80300190529722, 1.2404313985400175], [103.80396996287145, 1.243190652076852], [103.80589144884357, 1.2336185900997583], [103.80642530788192, 1.2388952544207388], [103.81017024691657, 1.244685947478631]]


In [ ]:
m = geemap.Map()
m.centerObject(AOI, 13)
m.add_basemap("SATELLITE")

m.addLayer(img, {"min": -30, "max": 0}, "VH (dB)")
m.addLayer(raw_ships, {"palette": ["orange"]}, "Raw CFAR hits")
m.addLayer(ships, {"palette": ["red"]}, "Filtered ship pixels")
m.addLayer(detections, {"color": "yellow"}, "Detections")
m.addLayer(ee.FeatureCollection(AOI).style(color="cyan", fillColor="00000000"), {}, "AOI")
m

Map(center=[1.2400002705199196, 103.84999999999863], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
import pandas as pd
pd.DataFrame(coords, columns=["lon", "lat"]).to_csv("detections.csv", index=False)


In [ ]:
%run -i ais_matching_v2.py

Detections read from `features`: 27
Scene start (UTC):    2026-09-01 22:50:00
Scene end   (UTC):    2026-09-01 22:50:25   (25 s long)
Matching time (mid):  2026-09-01 22:50:12
Pass: ASCENDING, incidence 42.9 deg, satellite heading 348 deg, looking toward 78 deg
Doppler correction ON: a ship moving 5 m/s (~10 kn) straight toward/away from the satellite shifts ~412 m
  note: could not read the scene properties ('system:time_start')
  note: scene end time unavailable: assumed a 25 s scene

!!! DEMO MODE: FAKE AIS data. The numbers below mean nothing. !!!

AIS check: data runs 11 min past the pass (good for interpolation).
RADAR LIST (everything SAR detected)
  Detections:                              27
    matched (score >= 0.5):                15
    possible dark (weak/ambiguous AIS):    6
        why: 6 next to an AIS ship already matched to another detection
    likely dark (no plausible AIS):        6
AIS LIST (ships broadcasting inside the AOI at the pass)
  AIS ships:             